In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch transformers peft accelerate psutil llama-cpp-python sentence-transformers pandas -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 26.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00


In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"]= "false"

In [4]:
import time
import torch
import psutil
import pandas as pd
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util

In [5]:
GGUF_MODEL = "/content/drive/MyDrive/model-q8_0.gguf"

PROMPTS = [
    """### Instruction:
Answer the HR question accurately.

### Input:
What is HR digitalization?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
Why is succession planning important?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
What is employee engagement?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
What is performance management?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
What is the role of HR in talent acquisition?

### Response:
"""
]
GROUND_TRUTH = [
    "HR digitalization is the use of digital tools and technologies to automate and improve HR processes such as recruitment, payroll, performance management, employee records, and employee engagement.",

    "Succession planning is important because it ensures business continuity by preparing employees to fill key roles, reduces leadership gaps, and supports long-term organizational stability and growth.",

    "Employee engagement refers to the emotional commitment employees have toward their organization, which influences their motivation, productivity, job satisfaction, and willingness to contribute to organizational goals.",

    "Performance management is a continuous process of setting goals, providing feedback, coaching, and evaluating employee performance to improve individual effectiveness and overall organizational outcomes.",

    "The role of HR in talent acquisition is to attract, screen, hire, and onboard suitable candidates while aligning recruitment strategies with business goals and workforce planning needs.",

    "Stay bonuses are financial or non-financial incentives offered to employees to encourage them to remain with an organization for a specified period, often used during mergers, restructuring, or critical project phases.",

    "Retention rates are calculated by dividing the number of employees who remain with the company over a specific period by the total number of employees at the start of that period, then multiplying by 100.",

    "When employees feel valued, it builds trust, increases morale, improves engagement, boosts productivity, and reduces turnover, leading to better performance and a healthier workplace culture.",

    "Onboarding is the structured process of integrating new hires into an organization by providing orientation, training, resources, and support to help them become productive and engaged employees.",

    "Workplace diversity refers to the inclusion of people from different backgrounds, cultures, genders, ages, abilities, and perspectives, which helps foster innovation, fairness, and better decision-making."
]


In [6]:
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def get_vram():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def accuracy(preds, refs):
    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)

    sims = util.cos_sim(p_emb, r_emb)

    per_sample_scores = sims.diag().cpu().numpy()

    # for i, score in enumerate(per_sample_scores):
    #     print(f"Sample {i+1} Similarity: {score:.3f}")

    mean_score = per_sample_scores.mean()

    return mean_score

In [8]:
def benchmark_gguf(label):
    llm = Llama(model_path=GGUF_MODEL, n_ctx=2048, n_threads=8, verbose=False)

    outputs = []
    start = time.time()

    for p in PROMPTS:
        response = ""
        stream = llm(p, max_tokens=256, stream=True)

        for output in stream:
            token = output["choices"][0]["text"]
            response += token
            # print(token, end="", flush=True)

        # print("\n \n")
        outputs.append(response)

    end = time.time()

    tokens = sum(len(o.split()) for o in outputs)
    tps = tokens / (end - start)
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(end - start, 2),
        "VRAM(MB)": 0,
        "Accuracy": round(acc, 3)
    }

results = []

results.append(benchmark_gguf("GGUF Q8 llama.cpp"))

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

import threading

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_MODEL = "/content/drive/MyDrive/merged_model"
GGUF_MODEL = "/content/drive/MyDrive/model-q8_0.gguf"
RESULTS_PATH = "./benchmarks/results.csv"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_PATH = "results.csv"

def benchmark_hf(model_path, label):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map=DEVICE)

    outputs = []
    start = time.time()

    for prompt in PROMPTS:
        streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        generation_kwargs = dict(
            **inputs,
            max_new_tokens=256,
            streamer=streamer,
            pad_token_id=tokenizer.eos_token_id
        )

        thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        response = ""
        for token in streamer:
            response += token
        outputs.append(response)
        thread.join()


    end = time.time()

    total_tokens = sum(len(tokenizer.encode(r)) for r in outputs)
    duration = end - start
    tps = total_tokens / duration
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(duration, 2),
        "VRAM(MB)": round(get_vram(), 2),
        "Accuracy": round(acc, 3)
    }
results.append(benchmark_hf(BASE_MODEL, "Base Model"))
results.append(benchmark_hf(FT_MODEL, "Fine-tuned"))

df = pd.DataFrame(results)
df.to_csv(RESULTS_PATH, index=False)

print(df)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

               Model  Tokens/sec  Latency(s)  VRAM(MB)  Accuracy
0  GGUF Q8 llama.cpp        3.17      243.79      0.00     0.814
1         Base Model       43.32       10.71   2534.16     0.864
2         Fine-tuned       33.75       43.14   2534.16     0.741


In [10]:
from llama_cpp import Llama

llm = Llama(
    model_path=GGUF_MODEL,
    n_ctx=2048
)

prompt = "Explain the role of HR in improving employee engagement."

print("Streaming output:\n")

for chunk in llm(prompt, max_tokens=120, stream=True):
    print(chunk["choices"][0]["text"], end="", flush=True)

llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from /content/drive/MyDrive/model-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   7:                  llama.feed_forward_l

Streaming output:



llama_perf_context_print:        load time =     726.08 ms
llama_perf_context_print: prompt eval time =     725.88 ms /    15 tokens (   48.39 ms per token,    20.66 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =     726.52 ms /    16 tokens
llama_perf_context_print:    graphs reused =          0


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def batch_inference(model_path):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

    prompts = [
        "What is employee onboarding?",
        "Why are performance reviews important?",
        "How can HR improve employee engagement?"
    ]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=120)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for i, out in enumerate(decoded):
        print(f"\nPrompt {i+1}:\n", out)

In [12]:
batch_inference(FT_MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Prompt 1:
 What is employee onboarding? 
Answer: Employee onboarding is a process of introducing new employees to the organization, its culture, policies, and processes. It helps new employees understand the organization, its goals, and how to achieve them.

Example: A new employee joins a financial institution. The onboarding process explains how to open a savings account, how to calculate interest, or how to resolve customer issues.

Why is employee onboarding important? 
Answer: Employee onboarding helps new employees feel comfortable and confident in their role. It also improves organizational outcomes

Prompt 2:
 Why are performance reviews important? 
Answer: Performance reviews are important in your career because they help you understand your strengths, weaknesses, and areas for improvement. They also help you make better decisions and take action towards your goals.

Example: If you are working on a project and you notice that you are not meeting the deadline, a performance r

In [13]:
!mkdir -p inference

In [14]:
%%writefile inference/test_inference.py

import time
import torch
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util


# =========================
# MODEL PATHS
# =========================

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_MODEL = "/content/drive/MyDrive/merged_model"
GGUF_MODEL = "/content/drive/MyDrive/model-q8_0.gguf"

RESULTS_PATH = "./benchmarks/results.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# TEST PROMPTS
# =========================

PROMPTS = [
    "What are the best practices for employee onboarding?",
    "How should performance reviews be conducted?",
    "What are strategies to improve employee engagement?"
]

GROUND_TRUTH = [
    "Effective employee onboarding includes clear communication of expectations, structured training, mentorship, and regular feedback to integrate employees successfully.",
    "Performance reviews should combine feedback, goal evaluation, constructive discussion, and development planning.",
    "Employee engagement can be improved through recognition, growth opportunities, open communication, and work-life balance."
]


# =========================
# EMBEDDING MODEL
# =========================

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")


# =========================
# VRAM CHECK
# =========================

def get_vram():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0


# =========================
# ACCURACY FUNCTION
# =========================

def accuracy(preds, refs):

    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)

    sims = util.cos_sim(p_emb, r_emb)

    return sims.diag().mean().item()


# =========================
# HF MODEL BENCHMARK
# =========================

def benchmark_hf(model_path, label):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

    outputs = []

    start = time.time()

    for prompt in PROMPTS:

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        out = model.generate(**inputs, max_new_tokens=128)

        text = tokenizer.decode(out[0], skip_special_tokens=True)

        outputs.append(text)

    end = time.time()

    tokens = sum(len(tokenizer.encode(o)) for o in outputs)

    tps = tokens / (end - start)

    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps,2),
        "Latency(s)": round(end-start,2),
        "VRAM(MB)": round(get_vram(),2),
        "Accuracy": round(acc,3)
    }


# =========================
# GGUF BENCHMARK
# =========================

def benchmark_gguf(label):

    llm = Llama(
        model_path=GGUF_MODEL,
        n_ctx=2048,
        n_threads=8,
        verbose=False
    )

    outputs = []

    start = time.time()

    for prompt in PROMPTS:

        out = llm(prompt, max_tokens=128)

        outputs.append(out["choices"][0]["text"])

    end = time.time()

    tokens = sum(len(o.split()) for o in outputs)

    tps = tokens / (end - start)

    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps,2),
        "Latency(s)": round(end-start,2),
        "VRAM(MB)": round(get_vram(),2),
        "Accuracy": round(acc,3)
    }


# =========================
# STREAMING OUTPUT DEMO
# =========================

def streaming_demo():

    print("\nStreaming Output Demo\n")

    llm = Llama(model_path=GGUF_MODEL, n_ctx=2048)

    prompt = "Explain the role of HR in improving employee engagement."

    for chunk in llm(prompt, max_tokens=120, stream=True):

        print(chunk["choices"][0]["text"], end="", flush=True)

    print("\n")


# =========================
# BATCH INFERENCE DEMO
# =========================

def batch_inference(model_path):

    print("\nBatch Inference Demo\n")

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

    prompts = [
        "What is employee onboarding?",
        "Why are performance reviews important?",
        "How can HR improve employee engagement?"
    ]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=120)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for i, out in enumerate(decoded):

        print(f"\nPrompt {i+1}:\n{out}")


# =========================
# MAIN
# =========================

if __name__ == "__main__":

    os.makedirs("benchmarks", exist_ok=True)

    results = []

    for i in range(3):

        results.append(benchmark_gguf("GGUF Q8 llama.cpp"))

        results.append(benchmark_hf(BASE_MODEL, "Base Model"))

        results.append(benchmark_hf(FT_MODEL, "Fine-tuned"))

    df = pd.DataFrame(results)

    df.to_csv(RESULTS_PATH, index=False)

    print("\nBenchmark Results\n")

    print(df)


    # Run streaming demo
    streaming_demo()


    # Run batch inference
    batch_inference(FT_MODEL)

Writing inference/test_inference.py


In [15]:
!python inference/test_inference.py

Loading weights: 100% 199/199 [00:00<00:00, 681.44it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 201/201 [00:08<00:00, 22.71it/s, Materializing param=model.norm.weight]
Loading weights: 100% 201/201 [00:09<00:00, 20.68it/s, Materializing param=model.norm.weight]
Loading weights: 100% 201/201 [00:08<00:00, 22.87it/s, Materializing param=model.norm.weight]
Loading weights: 100% 201/201 [00:09<00:00, 21.61it/s, Materializing param=model.norm.weight]
Loading weights: 100% 201/201 [00:08<00:00, 22.51it/s, Materializing param=model.norm.weight]
Loading weights: 100% 201/201 [00:09<00:00, 20.91it/s, Materializing param=model.norm.weight]

Benchmark Results

               Mode